## PydanticAI 제품 비교 리뷰 분석

실전_1_1에서 PydanticAI로 구조화한 리뷰 분석 결과를 활용하여, **3개 선크림 제품** 의 비교 분석을 수행합니다.

| 학습 목표 | 내용 |
|---|---|
| 구조화된 출력의 가치 | AI가 생성한 정형 데이터를 Python으로 자유롭게 가공 |
| 제품 비교 통계 | 항목별 긍정/부정 비율, 감정 분포, 장단점 집계 |
| 시각화 | seaborn/matplotlib로 비교 차트 생성 |
| AI 요약 | 비교 분석 결과를 PydanticAI로 한줄 요약 |

**선수 학습**: 실전_1_1 (리뷰 데이터 전처리)

### 환경 설정

In [ ]:
# ========================================
# 필요한 라이브러리를 불러옵니다
# ========================================

import os
import json
import platform
from collections import defaultdict, Counter
from itertools import combinations
from typing import List, Literal

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
from scipy.stats import chi2_contingency
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from pydantic_ai import Agent
from pydantic_ai.models.google import GoogleModelSettings
import plotly.graph_objects as go
import plotly.express as px
import networkx as nx
from pyvis.network import Network
from sklearn.feature_extraction.text import CountVectorizer

# 운영체제별 한글 폰트 설정
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
    korean_font_path = 'C:/Windows/Fonts/malgun.ttf'
elif platform.system() == 'Darwin':  # macOS
    plt.rcParams['font.family'] = 'AppleGothic'
    korean_font_path = '/System/Library/Fonts/AppleSDGothicNeo.ttc'
else:  # Linux
    plt.rcParams['font.family'] = 'NanumGothic'
    korean_font_path = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'

plt.rcParams['axes.unicode_minus'] = False

# .env 파일에서 API 키 로드
load_dotenv()
gemini_model = os.getenv('GEMINI_MODEL', 'gemini-3.1-flash-lite-preview')
model_id = f'google-gla:{gemini_model}'

# 비교 분석 결과 저장 폴더
output_dir = 'outputs/coupang_product_comparison'
os.makedirs(output_dir, exist_ok=True)

print(f"모델: {model_id}")
print(f"출력 폴더: {output_dir}/")

### 1. 분석 결과 로드

**실습 (1)**
- 실전_1_1에서 저장한 `product_review_analysis.json` 파일을 로드합니다.
- 구조화된 출력 덕분에 JSON을 DataFrame으로 바로 변환하여 분석할 수 있습니다.

In [ ]:
# =============================================================================
#  분석 결과 불러오기
# =============================================================================

analysis_file = os.path.join('outputs', 'product_review_analysis.json')

if not os.path.exists(analysis_file):
    print(f"[오류] {analysis_file} 파일이 없습니다.")
    print("실전_1_1_리뷰데이터_전처리.ipynb 를 먼저 실행해주세요.")
else:
    analysis_df = pd.read_json(analysis_file, orient='records')

    print(f"분석 결과 로드 완료: {len(analysis_df)}개 리뷰")
    print(f"컬럼: {list(analysis_df.columns)}")
    print()

    # 제품별 기본 통계
    print("제품별 리뷰 수:")
    for pid, group in analysis_df.groupby('product_id'):
        name = group.iloc[0]['product_name'][:40]
        print(f"  [{pid}] {name}... ({len(group)}개)")

    print()
    display(analysis_df[['product_name', 'rating', 'overall_sentiment', 'aspects', 'recommendation_score']].head())

### 2. 제품별 비교 통계

**실습 (2)**
- 제품별로 평점, 추천도, 감정 분포, 항목별 긍정/부정 비율, 장단점을 집계합니다.
- 한국어 `Literal` 값을 사용했으므로 영문 => 한글 매핑이 필요 없습니다.

In [ ]:
# =============================================================================
#  제품별 비교 통계 계산
#  - JSON에 pros/cons 필드가 없으므로,
#    각 aspect의 sentiment(긍정/부정)를 기준으로 장단점 키워드를 분류합니다.
#  - 긍정 aspect의 keywords → 장점(pros)
#  - 부정 aspect의 keywords → 단점(cons)
# =============================================================================

print("제품 비교 분석 시작...")
print()

# 워드클라우드에서 제외할 불용어
stopwords = {'없음', '있음', '사용', '좋음', '나쁨', '보통', '괜찮음', '추천', '만족', '불만족', '상품'}

product_stats = {}

for product_id, group in analysis_df.groupby('product_id'):
    product_name = group.iloc[0]['product_name']

    # 기본 통계
    stats = {
        'name': product_name,
        'review_count': len(group),
        'avg_rating': group['rating'].mean(),
        'rating_std': group['rating'].std() if len(group) > 1 else 0,
        'avg_recommendation': group['recommendation_score'].mean(),
        'sentiment_distribution': group['overall_sentiment'].value_counts().to_dict()
    }

    # 측면별 집계용 딕셔너리
    aspect_sentiment = defaultdict(list)                          # 측면별 감정 리스트
    aspect_count = {'pos': defaultdict(int), 'neg': defaultdict(int)}       # 측면별 긍정/부정 건수
    aspect_keywords = {'pos': defaultdict(list), 'neg': defaultdict(list)}  # 측면별 긍정/부정 키워드
    all_keywords = {'pos': [], 'neg': []}                         # 전체 장단점 키워드 (워드클라우드용)

    for _, review in group.iterrows():
        for aspect in review['aspects']:
            name = aspect['aspect']
            sentiment = aspect['sentiment']
            keywords = aspect.get('keywords', [])

            # 모든 감정 기록 (긍정 비율 계산에 사용)
            aspect_sentiment[name].append(sentiment)

            # 긍정/부정별 집계
            if sentiment == '긍정':
                aspect_count['pos'][name] += 1
                aspect_keywords['pos'][name].extend(keywords)
                all_keywords['pos'].extend(k for k in keywords if k.strip() not in stopwords)
            elif sentiment == '부정':
                aspect_count['neg'][name] += 1
                aspect_keywords['neg'][name].extend(keywords)
                all_keywords['neg'].extend(k for k in keywords if k.strip() not in stopwords)

    # 측면별 긍정 비율 (= 긍정 건수 / 전체 건수)
    aspect_scores = {
        name: sum(1 for s in sents if s == '긍정') / len(sents)
        for name, sents in aspect_sentiment.items() if sents
    }

    # stats에 결과 저장
    stats['aspect_scores'] = aspect_scores
    stats['aspect_positive'] = dict(aspect_count['pos'])
    stats['aspect_negative'] = dict(aspect_count['neg'])
    stats['aspect_keywords_pos'] = dict(aspect_keywords['pos'])
    stats['aspect_keywords_neg'] = dict(aspect_keywords['neg'])
    stats['top_pros'] = [item for item, _ in Counter(all_keywords['pos']).most_common(10)]
    stats['top_cons'] = [item for item, _ in Counter(all_keywords['neg']).most_common(10)]
    stats['all_pros'] = all_keywords['pos']
    stats['all_cons'] = all_keywords['neg']

    product_stats[product_id] = stats

# 결과 출력
for pid, stats in product_stats.items():
    print(f"[{pid}] {stats['name'][:40]}")
    print(f"  리뷰: {stats['review_count']}개 | 평균 평점: {stats['avg_rating']:.2f} | 추천도: {stats['avg_recommendation']:.2f}")
    print(f"  장점 (긍정 측면 키워드): {', '.join(stats['top_pros'][:3]) or '없음'}")
    print(f"  단점 (부정 측면 키워드): {', '.join(stats['top_cons'][:3]) or '없음'}")
    print()

In [ ]:
# =============================================================================
#  비교 요약 텍스트 생성 및 저장
# =============================================================================

summary_lines = []
summary_lines.append("제품 비교 요약:")
summary_lines.append("=" * 60)

for pid, stats in product_stats.items():
    summary_lines.append(f"\n제품: {stats['name'][:40]}")
    summary_lines.append(f"  리뷰 수: {stats['review_count']}개")
    summary_lines.append(f"  평균 평점: {stats['avg_rating']:.2f}점")
    summary_lines.append(f"  평균 추천도: {stats['avg_recommendation']:.2f}")
    summary_lines.append(f"  주요 장점: {', '.join(stats['top_pros'][:10])}")
    summary_lines.append(f"  주요 단점: {', '.join(stats['top_cons'][:10])}")

    if stats['aspect_scores']:
        summary_lines.append(f"  측면별 만족도 (긍정 비율):")
        sorted_aspects = sorted(stats['aspect_scores'].items(), key=lambda x: x[1], reverse=True)
        for aspect_name, score in sorted_aspects:
            summary_lines.append(f"    - {aspect_name}: {score:.2f} ({score*100:.1f}%)")

product_comparison_summary = "\n".join(summary_lines)
print(product_comparison_summary)

# 텍스트 파일로 저장
summary_path = f"{output_dir}/product_comparison_summary.txt"
with open(summary_path, 'w', encoding='utf-8') as f:
    f.write(product_comparison_summary)
print(f"\n저장 완료: {summary_path}")

### 3. 시각화

**실습 (3)**
- 제품별 평점/추천도 비교, 항목별 긍정 비율 히트맵, 감정 분포, 워드클라우드를 생성합니다.
- 모든 차트는 `analysis_outputs/` 폴더에 PNG로 저장됩니다.

In [ ]:
# =============================================================================
#  1. 평균 평점 및 추천도 비교
# =============================================================================

products = list(product_stats.keys())
product_names = [product_stats[pid]['name'][:20] + '...' for pid in products]

avg_ratings = [product_stats[pid]['avg_rating'] for pid in products]
rating_stds = [product_stats[pid]['rating_std'] for pid in products]
recommendations = [product_stats[pid]['avg_recommendation'] for pid in products]

fig, ax1 = plt.subplots(figsize=(10, 6))

x = np.arange(len(products))

# error bar 제거 — 바 위에 평점±표준편차 텍스트로 표시
bars = ax1.bar(x, avg_ratings, width=0.4, color='#5B9BD5', label='평균 평점')
ax1.set_ylabel('평점 (1~5점)', color='#5B9BD5')
ax1.set_ylim(0, 5.5)

for bar, val, std in zip(bars, avg_ratings, rating_stds):
    ax1.text(bar.get_x() + bar.get_width()/2, val + 0.1, f'{val:.2f} ±{std:.2f}',
             ha='center', va='bottom', fontweight='bold', fontsize=10, color='#5B9BD5')

ax2 = ax1.twinx()
ax2.plot(x, recommendations, 'o-', color='#ED7D31', linewidth=2, markersize=8, label='추천도')
for i, val in enumerate(recommendations):
    ax2.text(i, val - 0.06, f'{val:.2f}', ha='center', va='top', fontweight='bold', color='#ED7D31')
ax2.set_ylabel('추천도 (0~1)', color='#ED7D31')
ax2.set_ylim(0, 1.1)

ax1.set_xticks(x)
ax1.set_xticklabels(product_names, rotation=15, ha='right')
ax1.set_title('제품별 평균 평점 및 추천도 비교', fontsize=14, fontweight='bold')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='lower right',
           framealpha=0.9, edgecolor='gray')

plt.tight_layout()
plt.savefig(f"{output_dir}/rating_recommendation_comparison.png", dpi=150, bbox_inches='tight', pad_inches=0.2)
plt.show()

In [ ]:
# =============================================================================
#  2. 감정 분포 비교
# =============================================================================

sentiment_order = ['긍정', '혼합', '중립', '부정']
sentiment_colors = {'긍정': '#4CAF50', '혼합': '#FFC107', '중립': '#9E9E9E', '부정': '#F44336'}

pid_to_short = {pid: product_stats[pid]['name'][:20] + '...' for pid in products}
analysis_df['product_short'] = analysis_df['product_id'].map(pid_to_short)

sentiment_vals = [s for s in sentiment_order if s in analysis_df['overall_sentiment'].values]

fig, ax = plt.subplots(figsize=(10, 6))
sns.countplot(
    data=analysis_df, x='product_short', hue='overall_sentiment',
    hue_order=sentiment_vals,
    palette={s: sentiment_colors[s] for s in sentiment_vals},
    ax=ax
)
ax.set_title('제품별 감정 분포 비교', fontsize=14, fontweight='bold')
ax.set_xlabel('')
ax.set_ylabel('건수')
ax.legend(title='감정')

plt.tight_layout()
plt.savefig(f"{output_dir}/sentiment_distribution_comparison.png", dpi=150, bbox_inches='tight', pad_inches=0.2)
plt.show()

In [ ]:
# =============================================================================
#  3. 항목별 긍정 비율 히트맵
# =============================================================================

all_aspects = set()
for stats in product_stats.values():
    all_aspects.update(stats['aspect_scores'].keys())
all_aspects = sorted(list(all_aspects))

heatmap_data = []
for pid in products:
    row = [product_stats[pid]['aspect_scores'].get(asp, np.nan) for asp in all_aspects]
    heatmap_data.append(row)

heatmap_df = pd.DataFrame(heatmap_data, index=product_names, columns=all_aspects)

fig, ax = plt.subplots(figsize=(12, 5))
sns.heatmap(
    heatmap_df, annot=True, fmt='.2f', cmap='RdYlGn',
    vmin=0, vmax=1, center=0.5, linewidths=0.5,
    cbar_kws={'label': '긍정 비율'}, ax=ax
)
ax.set_title('제품별 항목 긍정 비율 히트맵', fontsize=14, fontweight='bold')
ax.set_ylabel('')

plt.tight_layout()
plt.savefig(f"{output_dir}/aspect_positive_heatmap.png", dpi=150, bbox_inches='tight', pad_inches=0.2)
plt.show()

In [ ]:
# =============================================================================
#  4. 제품별 레이더 차트: 항목별 긍정/부정 비율
# =============================================================================
import numpy as np

# 전체 제품에서 공통 항목 추출 (최소 3건 이상 데이터가 있는 항목만)
MIN_COUNT = 3
all_aspect_names = sorted(set(
    asp for stats in product_stats.values()
    for asp in stats['aspect_scores'].keys()
))

# 각 항목의 전체 언급 수 확인 -> 너무 적은 항목 제외
common_aspects = []
for asp in all_aspect_names:
    total = sum(
        stats['aspect_positive'].get(asp, 0) + stats['aspect_negative'].get(asp, 0)
        for stats in product_stats.values()
    )
    if total >= MIN_COUNT:
        common_aspects.append(asp)

n_aspects = len(common_aspects)

n_products = len(product_stats)
fig, axes = plt.subplots(1, n_products, figsize=(7 * n_products, 7),
                            subplot_kw=dict(polar=True))
if n_products == 1:
    axes = [axes]

# 레이더 차트 각도 계산
angles = np.linspace(0, 2 * np.pi, n_aspects, endpoint=False).tolist()
angles += angles[:1]

for idx, (pid, stats) in enumerate(product_stats.items()):
    ax = axes[idx]
    product_name = stats['name'][:25]

    # 항목별 긍정/부정 비율 계산
    pos_ratio = []
    neg_ratio = []
    for asp in common_aspects:
        pos = stats['aspect_positive'].get(asp, 0)
        neg = stats['aspect_negative'].get(asp, 0)
        total = pos + neg
        if total > 0:
            pos_ratio.append(pos / total)
            neg_ratio.append(neg / total)
        else:
            pos_ratio.append(0)
            neg_ratio.append(0)

    pos_closed = pos_ratio + [pos_ratio[0]]
    neg_closed = neg_ratio + [neg_ratio[0]]

    # 긍정 비율 (초록)
    ax.fill(angles, pos_closed, alpha=0.2, color='#4CAF50')
    ax.plot(angles, pos_closed, color='#4CAF50', linewidth=2, label='긍정 비율')
    ax.scatter(angles[:-1], pos_ratio, color='#4CAF50', s=40, zorder=5)

    # 부정 비율 (빨강)
    ax.fill(angles, neg_closed, alpha=0.2, color='#F44336')
    ax.plot(angles, neg_closed, color='#F44336', linewidth=2, label='부정 비율')
    ax.scatter(angles[:-1], neg_ratio, color='#F44336', s=40, zorder=5)

    # 축 설정
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(common_aspects, fontsize=10)
    ax.set_ylim(0, 1)
    ax.set_yticks([0.25, 0.5, 0.75, 1.0])
    ax.set_yticklabels(['25%', '50%', '75%', '100%'], fontsize=8, color='grey')
    ax.set_title(product_name, fontsize=13, fontweight='bold', pad=20)
    ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=9)

plt.suptitle('제품별 항목 긍정/부정 비율 (레이더 차트)', fontsize=15, fontweight='bold', y=1.03)
plt.tight_layout()
plt.savefig(f"{output_dir}/product_radar_charts.png", dpi=150, bbox_inches='tight', pad_inches=0.3)
plt.show()

In [ ]:
# =============================================================================
#  5. 제품별 측면별 긍정/부정 비율 (스택 막대 차트)
# =============================================================================

for pid, stats in product_stats.items():
    product_name = stats['name'][:40]

    # 해당 제품의 측면들
    aspects = sorted(set(list(stats['aspect_positive'].keys()) + list(stats['aspect_negative'].keys())))

    if aspects:
        positive_counts = [stats['aspect_positive'].get(asp, 0) for asp in aspects]
        negative_counts = [stats['aspect_negative'].get(asp, 0) for asp in aspects]

        # 만족도(긍정 비율) 계산
        satisfaction_ratios = []
        for asp in aspects:
            pos = stats['aspect_positive'].get(asp, 0)
            neg = stats['aspect_negative'].get(asp, 0)
            total = pos + neg
            ratio = (pos / total * 100) if total > 0 else 0
            satisfaction_ratios.append(f'{ratio:.1f}%')

        fig = go.Figure()

        # 긍정 막대 (아래)
        fig.add_trace(go.Bar(
            name='긍정', x=aspects, y=positive_counts,
            marker_color='lightgreen',
            text=positive_counts, textposition='inside'
        ))

        # 부정 막대 (위에 스택)
        fig.add_trace(go.Bar(
            name='부정', x=aspects, y=negative_counts,
            marker_color='lightcoral',
            text=negative_counts, textposition='inside'
        ))

        # 만족도 텍스트 (막대 위)
        total_counts = [p + n for p, n in zip(positive_counts, negative_counts)]
        fig.add_trace(go.Scatter(
            x=aspects,
            y=[total + max(total_counts) * 0.05 for total in total_counts],
            mode='text',
            text=[f'만족도: {r}' for r in satisfaction_ratios],
            textposition='top center',
            textfont=dict(size=11, color='black', family='Arial Black'),
            showlegend=False
        ))

        fig.update_layout(
            title=f'{product_name} - 측면별 긍정/부정 비교',
            xaxis_title='분석 측면', yaxis_title='리뷰 수',
            barmode='stack', height=500, showlegend=True
        )
        fig.show()

In [ ]:
# =============================================================================
#  6. 제품별 측면-감정 트리맵
# =============================================================================
print("""트리맵 해석: 사각형의 크기는 리뷰 개수, 색상은 감정(초록=긍정, 노랑=중립, 빨강=부정)을 나타냅니다.
각 측면 안의 긍정/부정/중립 비율을 한눈에 비교할 수 있습니다.
""")

sentiment_color_map = {'긍정': 1, '중립': 0, '부정': -1}

for pid, stats in product_stats.items():
    product_name = stats['name'][:30]

    # 트리맵 데이터 준비
    labels = [product_name]
    parents = ['']
    values = [0]
    colors = [0]

    # 측면별 데이터 수집
    aspect_data = defaultdict(lambda: {'긍정': 0, '중립': 0, '부정': 0})

    for _, review in analysis_df[analysis_df['product_id'] == pid].iterrows():
        if isinstance(review['aspects'], list):
            for aspect_analysis in review['aspects']:
                if isinstance(aspect_analysis, dict):
                    aspect = aspect_analysis.get('aspect')
                    sentiment = aspect_analysis.get('sentiment')
                    if aspect and sentiment:
                        if sentiment in sentiment_color_map:
                            aspect_data[aspect][sentiment] += 1

    # 측면별 노드 추가
    for aspect, sentiments in aspect_data.items():
        aspect_total = sum(sentiments.values())
        if aspect_total == 0:
            continue

        aspect_avg = sum(sentiment_color_map.get(s, 0) * cnt
                         for s, cnt in sentiments.items()) / aspect_total

        labels.append(aspect)
        parents.append(product_name)
        values.append(aspect_total)
        colors.append(aspect_avg)

        # 감정별 세부 노드
        for sentiment, count in sentiments.items():
            if count > 0:
                labels.append(f'{aspect}-{sentiment}')
                parents.append(aspect)
                values.append(count)
                colors.append(sentiment_color_map.get(sentiment, 0))

    if len(labels) > 1:
        fig = px.treemap(
            names=labels, parents=parents, values=values,
            color=colors, color_continuous_scale='RdYlGn',
            color_continuous_midpoint=0,
            title=f'{product_name} - 측면별 감정 분포 (트리맵)'
        )
        fig.update_traces(
            textposition='middle center',
            textfont=dict(size=12),
            hovertemplate='<b>%{label}</b><br>리뷰 수: %{value}<extra></extra>'
        )
        fig.update_layout(height=600, margin=dict(t=50, l=25, r=25, b=25))
        fig.show()

In [ ]:
# =============================================================================
#  7. 제품별 장단점 워드클라우드 (긍정/부정 측면 키워드 기반)
# =============================================================================

for pid, stats in product_stats.items():
    product_name = stats['name'][:30]

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    pros_text = ' '.join(stats['all_pros']) if stats['all_pros'] else '데이터없음'
    wc_pros = WordCloud(
        font_path=korean_font_path, width=600, height=400,
        background_color='white', colormap='Greens',
        stopwords=stopwords, max_words=50
    ).generate(pros_text)
    axes[0].imshow(wc_pros, interpolation='bilinear')
    axes[0].set_title('장점 (긍정 측면 키워드)', fontsize=12, fontweight='bold', color='#4CAF50', pad=10)
    axes[0].axis('off')

    cons_text = ' '.join(stats['all_cons']) if stats['all_cons'] else '데이터없음'
    wc_cons = WordCloud(
        font_path=korean_font_path, width=600, height=400,
        background_color='white', colormap='Reds',
        stopwords=stopwords, max_words=50
    ).generate(cons_text)
    axes[1].imshow(wc_cons, interpolation='bilinear')
    axes[1].set_title('단점 (부정 측면 키워드)', fontsize=12, fontweight='bold', color='#F44336', pad=10)
    axes[1].axis('off')

    fig.suptitle(f'{product_name} - 장단점 워드클라우드', fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(f"{output_dir}/wordcloud_{pid}.png", dpi=150, bbox_inches='tight', pad_inches=0.2)
    plt.show()

In [ ]:
# =============================================================================
#  8. 제품별 항목별 긍정/부정 키워드 워드클라우드
# =============================================================================

for pid, stats in product_stats.items():
    product_name = stats['name'][:30]

    aspects_with_data = []
    for asp in stats['aspect_keywords_pos'].keys() | stats['aspect_keywords_neg'].keys():
        if stats['aspect_keywords_pos'].get(asp, []) or stats['aspect_keywords_neg'].get(asp, []):
            aspects_with_data.append(asp)

    if not aspects_with_data:
        continue

    n_aspects = len(aspects_with_data)
    fig, axes = plt.subplots(n_aspects, 2, figsize=(14, 3.5 * n_aspects))
    if n_aspects == 1:
        axes = axes.reshape(1, -1)

    for idx, asp in enumerate(aspects_with_data):
        pos_text = ' '.join(stats['aspect_keywords_pos'].get(asp, [])) or '데이터없음'
        wc = WordCloud(
            font_path=korean_font_path, width=400, height=200,
            background_color='white', colormap='Greens', max_words=30
        ).generate(pos_text)
        axes[idx, 0].imshow(wc, interpolation='bilinear')
        axes[idx, 0].set_title(f'{asp} - 긍정', fontsize=10, color='#4CAF50', pad=12)
        axes[idx, 0].axis('off')

        neg_text = ' '.join(stats['aspect_keywords_neg'].get(asp, [])) or '데이터없음'
        wc = WordCloud(
            font_path=korean_font_path, width=400, height=200,
            background_color='white', colormap='Reds', max_words=30
        ).generate(neg_text)
        axes[idx, 1].imshow(wc, interpolation='bilinear')
        axes[idx, 1].set_title(f'{asp} - 부정', fontsize=10, color='#F44336', pad=12)
        axes[idx, 1].axis('off')

    plt.suptitle(f'{product_name} - 항목별 키워드', fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.subplots_adjust(hspace=0.4)
    plt.savefig(f"{output_dir}/aspect_keywords_{pid}.png", dpi=150, bbox_inches='tight', pad_inches=0.3)
    plt.show()

In [ ]:
# =============================================================================
#  9. 긍정/부정 키워드 동시출현(Co-occurrence) 분석
# =============================================================================

# ── 네트워크 시각화 함수 ──
def create_network_graph(pairs, filename, title="", top_n=30,
                         node_color="#3498db", height="700px"):
    """동시출현 쌍으로 pyvis 인터랙티브 네트워크 HTML을 생성합니다."""
    G = nx.Graph()
    for w1, w2, count in pairs[:top_n]:
        G.add_edge(w1, w2, weight=int(count))

    net = Network(height=height, width="100%", bgcolor="#ffffff",
                  font_color="#333333")

    node_weights = {}
    for node in G.nodes():
        node_weights[node] = sum(d["weight"] for _, _, d in G.edges(node, data=True))

    max_nw = max(node_weights.values())
    min_nw = min(node_weights.values())
    nw_range = max_nw - min_nw if max_nw != min_nw else 1

    for node in G.nodes():
        normalized_size = 15 + (node_weights[node] - min_nw) / nw_range * 35
        net.add_node(node, label=node,
                     size=normalized_size,
                     color=node_color,
                     font={"size": 16, "face": "Malgun Gothic"})

    max_ew = max(d["weight"] for _, _, d in G.edges(data=True))

    for u, v, d in G.edges(data=True):
        w = d["weight"]
        net.add_edge(u, v,
                     width=1 + (w / max_ew) * 7,
                     color={"color": "#aaaaaa", "opacity": 0.6},
                     title=f"{u} + {v}: {w}회")

    net.set_options('{"physics":{"forceAtlas2Based":{"gravitationalConstant":-80,"centralGravity":0.01,"springLength":150,"springConstant":0.05},"solver":"forceAtlas2Based","stabilization":{"iterations":200}},"interaction":{"hover":true,"zoomView":true,"dragNodes":true}}')

    net.save_graph(filename)

    if title:
        with open(filename, "r", encoding="utf-8") as f:
            html = f.read()
        title_tag = f'<h2 style="text-align:center; font-family:Malgun Gothic; color:#333;">{title}</h2>'
        html = html.replace("<body>", f"<body>\n{title_tag}", 1)
        with open(filename, "w", encoding="utf-8") as f:
            f.write(html)

    print(f"  => {filename} 저장 완료" + (f" ({title})" if title else ""))


# ── 제품별 긍정/부정 키워드 동시출현 분석 ──
for pid, stats in product_stats.items():
    product_name = stats["name"][:30]
    print(f"\n{'='*60}")
    print(f" {product_name}")
    print(f"{'='*60}")

    for sentiment_label, keywords_list, bar_color, net_color in [
        ("긍정", stats["all_pros"], "mediumseagreen", "#2ecc71"),
        ("부정", stats["all_cons"], "indianred", "#e74c3c"),
    ]:
        if len(keywords_list) < 5:
            print(f"\n  [{sentiment_label}] 키워드가 부족하여 건너뜁니다.")
            continue

        # 측면별 키워드를 문서 단위로 구성 (동시출현 계산용)
        key = "pos" if sentiment_label == "긍정" else "neg"
        aspect_kw = stats[f"aspect_keywords_{key}"]
        review_keyword_docs = []
        for asp, kws in aspect_kw.items():
            if kws:
                review_keyword_docs.append(" ".join(kws))

        # 문서가 부족하면 키워드를 5개씩 묶어서 가상 문서 생성
        if len(review_keyword_docs) < 3:
            kw_list = keywords_list
            review_keyword_docs = [" ".join(kw_list[i:i+5]) for i in range(0, len(kw_list), 5)]

        if len(review_keyword_docs) < 2:
            print(f"\n  [{sentiment_label}] 문서가 부족하여 건너뜁니다.")
            continue

        # CountVectorizer로 DTM 생성
        co_vec = CountVectorizer(ngram_range=(1, 1), max_df=0.95, min_df=1, binary=True)
        try:
            co_dtm = co_vec.fit_transform(review_keyword_docs)
        except ValueError:
            print(f"\n  [{sentiment_label}] 유효한 키워드가 없어 건너뜁니다.")
            continue
        co_words = co_vec.get_feature_names_out()

        # 동시출현 행렬
        co_matrix = (co_dtm.T @ co_dtm)
        co_matrix.setdiag(0)
        co_array = co_matrix.toarray()

        # 상위 동시출현 쌍 추출
        co_pairs = []
        for i in range(len(co_words)):
            for j in range(i + 1, len(co_words)):
                if co_array[i, j] > 0:
                    co_pairs.append((co_words[i], co_words[j], co_array[i, j]))
        co_pairs.sort(key=lambda x: x[2], reverse=True)

        if not co_pairs:
            print(f"\n  [{sentiment_label}] 동시출현 쌍이 없습니다.")
            continue

        # ── 막대 그래프 ──
        top_pairs = co_pairs[:15]
        pair_labels = [f"{w1} + {w2}" for w1, w2, _ in top_pairs]
        pair_counts = [int(c) for _, _, c in top_pairs]

        fig, ax = plt.subplots(figsize=(10, 7))
        co_plot_df = pd.DataFrame({"단어 쌍": pair_labels, "동시출현 횟수": pair_counts})
        sns.barplot(data=co_plot_df, x="동시출현 횟수", y="단어 쌍", color=bar_color, ax=ax)
        ax.set_title(f"{product_name} - {sentiment_label} 키워드 동시출현 상위 15개",
                     fontsize=13, fontweight="bold")
        plt.tight_layout()
        plt.savefig(f"{output_dir}/cooccurrence_{sentiment_label}_{pid}.png",
                    dpi=150, bbox_inches="tight", pad_inches=0.2)
        plt.show()

        # ── 네트워크 그래프 ──
        create_network_graph(
            pairs=co_pairs,
            filename=f"{output_dir}/cooccurrence_network_{sentiment_label}_{pid}.html",
            title=f"{product_name} - {sentiment_label} 키워드 연관성 네트워크",
            top_n=25,
            node_color=net_color,
            height="600px"
        )

print("\n=> 각 HTML 파일을 브라우저에서 열어 네트워크를 탐색해보세요.")

### 5. AI 비교 요약 생성

**실습 (5)**
- 비교 분석 통계 텍스트를 PydanticAI Agent에 입력하여, **구조화된 비교 요약** 을 생성합니다.
- 이것이 구조화된 출력의 핵심 가치입니다: AI가 생성한 정형 데이터를 코드로 가공한 뒤, 다시 AI에 입력하여 새로운 인사이트를 얻습니다.

In [ ]:
# =============================================================================
#  AI 비교 요약 생성 (텍스트 + 시각화 이미지 멀티모달 분석)
# =============================================================================
import glob

# ── 스키마 설계 ──

class AspectAnalysis(BaseModel):
    """측면별 분석"""
    aspect_name: str = Field(description="분석 측면 (예: 사용성, 품질, 가성비)")
    sentiment_ratio: str = Field(description="긍정/부정 비율 요약 (예: 긍정 85%, 부정 15%)")
    key_keywords: List[str] = Field(description="해당 측면의 핵심 키워드 3~5개")
    insight: str = Field(description="이 측면에서의 핵심 인사이트", max_length=150)

class ProductProfile(BaseModel):
    """개별 제품 프로필"""
    product_name: str = Field(description="제품명")
    strengths: List[str] = Field(description="주요 강점 (3~5개, 차트 데이터 근거 포함)")
    weaknesses: List[str] = Field(description="주요 약점 (2~3개, 차트 데이터 근거 포함)")
    aspect_analyses: List[AspectAnalysis] = Field(description="측면별 상세 분석 (상위 3~5개 측면)")
    best_for: str = Field(description="이 제품이 가장 적합한 사용자/상황", max_length=150)
    caution_for: str = Field(description="이 제품이 부적합한 사용자/상황", max_length=150)

class VisualizationInsight(BaseModel):
    """시각화 차트에서 발견한 인사이트"""
    chart_name: str = Field(description="차트 이름 (예: 평점 비교, 감정 분포, 레이더 차트)")
    finding: str = Field(description="차트에서 발견한 핵심 발견", max_length=200)
    implication: str = Field(description="이 발견이 소비자에게 의미하는 것", max_length=200)

class ComparisonSummary(BaseModel):
    """제품 비교 종합 요약"""
    profiles: List[ProductProfile] = Field(description="각 제품의 상세 프로필")
    visualization_insights: List[VisualizationInsight] = Field(
        description="시각화 차트에서 도출한 핵심 인사이트 (3~5개)"
    )
    key_differences: List[str] = Field(description="제품 간 핵심 차이점 (3~5개)")
    keyword_association_insight: str = Field(
        description="동시출현 네트워크에서 발견한 키워드 연관성 인사이트",
        max_length=300
    )
    overall_recommendation: str = Field(
        description="상황별 추천 가이드 (피부타입, 용도, 예산 등 기준별)",
        max_length=400
    )
    one_line_summary: str = Field(description="전체 비교 결과 한줄 요약", max_length=200)


# ── 이미지 수집 ──
image_files = sorted(glob.glob(f"{output_dir}/*.png"))
print(f"첨부할 이미지: {len(image_files)}개")
for f in image_files:
    print(f"  - {os.path.basename(f)}")

# 이미지를 BinaryContent로 변환
image_contents = []
for img_path in image_files:
    with open(img_path, "rb") as f:
        image_contents.append(
            BinaryContent(data=f.read(), media_type="image/png")
        )


# ── Agent 생성 및 실행 ──
summary_agent = Agent(
    model_id,
    output_type=ComparisonSummary,
    system_prompt=(
        "당신은 제품 비교 분석 전문가입니다. "
        "텍스트 통계 데이터와 시각화 차트 이미지를 모두 분석하여 종합적인 비교 요약을 작성합니다. "
        "차트에서 보이는 패턴, 분포, 상대적 차이를 구체적으로 언급하세요. "
        "순위보다는 제품별 특성과 상황별 추천에 초점을 맞추세요."
    ),
)

prompt = f"""다음 제품 비교 분석 결과(텍스트 통계 + 시각화 차트)를 종합하여 비교 요약을 작성해주세요.

## 텍스트 통계 데이터
{product_comparison_summary}

## 첨부된 시각화 차트 설명
- 평균 평점 및 추천도 비교 막대/라인 차트
- 감정 분포 비교 차트
- 항목별 긍정 비율 히트맵
- 제품별 레이더 차트 (긍정/부정 비율)
- 측면별 긍정/부정 스택 막대 차트
- 장단점 워드클라우드
- 긍정/부정 키워드 동시출현 막대 그래프

각 차트에서 보이는 시각적 패턴과 텍스트 데이터를 교차 분석하여,
제품별 강점/약점, 핵심 차이점, 상황별 추천을 작성해주세요."""

result = await summary_agent.run(
    [prompt] + image_contents,
    model_settings=GoogleModelSettings(temperature=0.3)
)
comparison = result.output

# ── 결과 출력 ──
print("=" * 60)
print("  AI 비교 요약 (텍스트 + 이미지 멀티모달 분석)")
print("=" * 60)
print()
for p in comparison.profiles:
    print(f"■ {p.product_name}")
    print(f"  강점: {chr(10).join(f"    - {s}" for s in p.strengths)}")
    print(f"  약점: {chr(10).join(f"    - {w}" for w in p.weaknesses)}")
    print(f"  측면별 분석:")
    for a in p.aspect_analyses:
        print(f"    [{a.aspect_name}] {a.sentiment_ratio} | 키워드: {
.join(a.key_keywords)}")
        print(f"      → {a.insight}")
    print(f"  추천 대상: {p.best_for}")
    print(f"  비추천 대상: {p.caution_for}")
    print()

print("▶ 시각화 인사이트:")
for vi in comparison.visualization_insights:
    print(f"  [{vi.chart_name}] {vi.finding}")
    print(f"    → {vi.implication}")
print()
print("▶ 핵심 차이점:")
for diff in comparison.key_differences:
    print(f"  - {diff}")
print()
print(f"▶ 키워드 연관성 인사이트: {comparison.keyword_association_insight}")
print()
print(f"▶ 상황별 추천: {comparison.overall_recommendation}")
print(f"▶ 한줄 요약: {comparison.one_line_summary}")

### 결과 저장

생성된 파일들을 확인합니다. 이 파일들은 다음 노트북(실전_1_3)에서 종합 인사이트 도출의 입력 데이터로 사용됩니다.

In [ ]:
# =============================================================================
#  AI 비교 요약을 product_comparison_summary.txt에 추가 저장
# =============================================================================

ai_lines = []
ai_lines.append("")
ai_lines.append("")
ai_lines.append("=" * 60)
ai_lines.append("AI 비교 요약 (텍스트 + 이미지 멀티모달 분석)")
ai_lines.append("=" * 60)

for p in comparison.profiles:
    ai_lines.append(f"\n\u25a0 {p.product_name}")
    ai_lines.append(f"  강점:")
    for s in p.strengths:
        ai_lines.append(f"    - {s}")
    ai_lines.append(f"  약점:")
    for w in p.weaknesses:
        ai_lines.append(f"    - {w}")
    ai_lines.append(f"  측면별 분석:")
    for a in p.aspect_analyses:
        ai_lines.append(f"    [{a.aspect_name}] {a.sentiment_ratio} | 키워드: {', '.join(a.key_keywords)}")
        ai_lines.append(f"      → {a.insight}")
    ai_lines.append(f"  추천 대상: {p.best_for}")
    ai_lines.append(f"  비추천 대상: {p.caution_for}")

ai_lines.append("")
ai_lines.append("시각화 인사이트:")
for vi in comparison.visualization_insights:
    ai_lines.append(f"  [{vi.chart_name}] {vi.finding}")
    ai_lines.append(f"    → {vi.implication}")

ai_lines.append("")
ai_lines.append("핵심 차이점:")
for diff in comparison.key_differences:
    ai_lines.append(f"  - {diff}")

ai_lines.append(f"\n키워드 연관성 인사이트: {comparison.keyword_association_insight}")
ai_lines.append(f"\n상황별 추천: {comparison.overall_recommendation}")
ai_lines.append(f"한줄 요약: {comparison.one_line_summary}")

summary_path = f"{output_dir}/product_comparison_summary.txt"
with open(summary_path, "a", encoding="utf-8") as f:
    f.write("\n".join(ai_lines))
print(f"AI 비교 요약이 {summary_path}에 추가 저장되었습니다.")